# 02b - Huấn luyện lại (TF Flowers + ảnh tự thu thập) + lớp "Khác" (mô hình 6 lớp, v2-khac)

Người phụ trách: Hân

Tuần 13–14: giống v2 (notebook 02) nhưng thêm lớp Khác, để so với v1-khac (notebook 01b).
- 5 lớp hoa: dữ liệu của v2 – `combined.zip` (train, val) và `web_test.zip` (ảnh thực tế) do `data_tool/split.py` tạo.
  Tập test 5 loài vẫn là TF test như v1, nên so được trực tiếp.
- Lớp Khác: **cùng quy tắc với 01b** (cỡ trung bình 1 loài hoa, nửa hoa ngoài 5 loài / nửa không phải hoa).
  Loài hoa ở v2 nhiều ảnh hơn nên lớp Khác cũng lớn hơn; lưu riêng thành `other_split_v2.zip`.

Lớp `other` ("Khác") gồm 2 phần, lấy từ các bộ dữ liệu công khai (tải bản sao trên Hugging Face, vì máy chủ gốc
của Oxford không truy cập được và máy chủ Caltech chặn tải bằng Python):
- **Hoa ngoài 5 loài**: Oxford 102 Flowers (bỏ các loài trùng hoặc rất giống 5 loài của Anthea).
- **Không phải hoa**: Caltech-101 (người, động vật, xe cộ, đồ vật…) và Imagenette (nhà thờ, xe rác, cây xăng…).

Kết quả lưu vào `models_khac_v2/` trên Drive, **không ghi đè** `models/` (v2) và `models_khac/` (v1-khac).

Cell 1 – Thư viện, kiểm tra GPU, cố định seed

In [ ]:
import os
import io
import json
import math
import shutil
import random
import datetime

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import tensorflow as tf
from PIL import Image
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import regularizers
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

print('TensorFlow:', tf.__version__)

gpus = tf.config.list_physical_devices('GPU')
if len(gpus) > 0:
    print('GPU:', gpus)
else:
    print('GPU: KHÔNG CÓ – chọn Runtime → Change runtime type → T4 GPU')

SEED = 123
keras.utils.set_random_seed(SEED)

Cell 2 – Kết nối Google Drive, khai báo đường dẫn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ROOT         = '/content/drive/MyDrive/flower-classifier'
MODEL_DIR    = f'{ROOT}/models_khac_v2'                # KHÔNG ghi đè models/ (v2) và models_khac/ (v1-khac)
CKPT_PATH    = f'{ROOT}/checkpoints/best_khac_v2.keras'
TF_ZIP       = f'{ROOT}/data/tf_flowers_split.zip'     # notebook 01: TF Flowers chia 70/15/15 (lấy phần test)
COMBINED_ZIP = f'{ROOT}/data/combined.zip'             # split.py: TF + ảnh web (train, val) – dữ liệu của v2
WEB_TEST_ZIP = f'{ROOT}/data/web_test.zip'             # split.py: ảnh web (test) – "ảnh thực tế"
OTHER_ZIP    = f'{ROOT}/data/other_split_v2.zip'       # Cell 4 tạo lần đầu; lần sau giải nén lại cho giống hệt.
                                                       # Khác other_split.zip của 01b (lớp Khác của v1)

for d in [MODEL_DIR, f'{ROOT}/checkpoints', f'{ROOT}/data']:
    os.makedirs(d, exist_ok=True)

FLOWERS     = ['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']
CLASS_NAMES = FLOWERS + ['other']    # "other" ở CUỐI: 5 loài giữ đúng số thứ tự 0–4 như v1, v2
VERSION     = 'v2-khac'
THRESHOLD   = 0.5                    # ngưỡng "chưa chắc chắn", giống v1, v2
IMG_SIZE    = (180, 180)
BATCH_SIZE  = 32

Cell 3 – Giải nén dữ liệu của v2 (5 loài hoa)

In [ ]:
# Giải nén lại từ đầu: xóa luôn thư mục other/ mà Cell 4 của lần chạy trước đã chép vào
shutil.rmtree('/content/tf_flowers_split', ignore_errors=True)
shutil.unpack_archive(TF_ZIP, '/content/tf_flowers_split')   # -> /content/tf_flowers_split/{train,val,test}/<loài>
shutil.rmtree('/content/combined', ignore_errors=True)
shutil.unpack_archive(COMBINED_ZIP, '/content')               # -> /content/combined/{train,val}/<loài>
shutil.rmtree('/content/web_split', ignore_errors=True)
shutil.unpack_archive(WEB_TEST_ZIP, '/content')               # -> /content/web_split/test/<loài>

TRAIN_DIR    = '/content/combined/train'
VAL_DIR      = '/content/combined/val'
TEST_DIR     = '/content/tf_flowers_split/test'   # cùng tập TF test đã chấm v1, v2, v1-khac
WEB_TEST_DIR = '/content/web_split/test'

def count(folder):
    """Đếm số ảnh trong từng thư mục con. Mỗi thư mục con là 1 lớp (daisy, roses, ...)."""
    result = {}
    for class_name in sorted(os.listdir(folder)):
        result[class_name] = len(os.listdir(f'{folder}/{class_name}'))
    return result


print('train    ', count(TRAIN_DIR))
print('val      ', count(VAL_DIR))
print('TF test  ', count(TEST_DIR))
print('web test ', count(WEB_TEST_DIR))

Cell 4 – Tạo lớp Khác cho v2 (cùng cách với 01b, cỡ theo loài hoa của v2), chia cố định 70/15/15

In [ ]:
# Hoa trùng hoặc rất giống 5 loài của Anthea: bỏ, để lớp Khác không chứa ảnh mà người xem cũng gọi là cúc, hồng…
OXFORD_SKIP = ['oxeye daisy', 'barbeton daisy', 'common dandelion', "colt's foot",
               'rose', 'sunflower', 'black-eyed susan']
# Caltech-101: bỏ 3 nhóm là hoa, và background_google (ảnh ngẫu nhiên, có thể lẫn hoa)
CALTECH_SKIP = ['lotus', 'sunflower', 'water_lilly', 'background_google']
OTHER_DIR = '/content/other_split'

HF_API  = 'https://huggingface.co/api/datasets'
HF_INFO = 'https://datasets-server.huggingface.co/info?dataset='


def load_by_class(dataset, skip):
    """Tải một bộ ảnh trên Hugging Face (dạng file parquet).
    Trả về dict: {tên lớp con: [ảnh 1 (bytes), ảnh 2 (bytes), ...]}, đã bỏ các lớp con nằm trong skip."""
    # Bước 1: lấy tên các lớp con. names[0] là tên của nhãn số 0, names[1] là tên của nhãn số 1, ...
    info = requests.get(HF_INFO + dataset).json()
    names = info['dataset_info']['default']['features']['label']['names']

    # Bước 2: lấy link các file parquet, dạng {'train': [link, ...], 'test': [link, ...]}
    files = requests.get(f'{HF_API}/{dataset}/parquet/default').json()

    # Bước 3: tải từng file, xếp từng ảnh vào đúng lớp con. Gộp mọi tập con (train, validation, test)
    by_class = {}
    for split_name in files:
        for link in files[split_name]:
            response = requests.get(link)
            response.raise_for_status()                              # tải lỗi thì dừng luôn, báo lỗi
            table = pd.read_parquet(io.BytesIO(response.content))   # bảng: mỗi dòng có cột image và cột label

            for image, label in zip(table['image'], table['label']):
                class_name = names[label].strip()
                if class_name in skip:
                    continue                      # lớp con bị bỏ -> bỏ qua ảnh này
                if class_name not in by_class:
                    by_class[class_name] = []     # gặp lớp con mới -> tạo danh sách rỗng
                by_class[class_name].append(image['bytes'])
    return by_class


def sample_evenly(by_class, n, rng):
    """Lấy n ảnh, chia đều cho các lớp con (mỗi lớp con vài tấm) để lớp Khác đa dạng nhất."""
    per_class = math.ceil(n / len(by_class))     # số ảnh mỗi lớp con, làm tròn lên
    picked = []
    for class_name in sorted(by_class):
        images = by_class[class_name]
        k = min(per_class, len(images))          # lớp con ít ảnh hơn per_class thì lấy hết
        picked += rng.sample(images, k)          # chọn ngẫu nhiên k ảnh, không trùng
    rng.shuffle(picked)
    return picked[:n]                            # làm tròn lên nên có thể dư vài ảnh -> cắt còn đúng n

if os.path.exists(OTHER_ZIP):
    # Đã tạo ở lần chạy trước -> giải nén lại, lớp Khác giống hệt lần trước
    shutil.rmtree(OTHER_DIR, ignore_errors=True)
    shutil.unpack_archive(OTHER_ZIP, OTHER_DIR)
else:
    shutil.rmtree(OTHER_DIR, ignore_errors=True)   # bắt đầu từ thư mục trống, không lẫn ảnh của lần chạy trước

    # Lớp Khác cỡ trung bình 5 lớp hoa CỦA V2: số ảnh train trung bình / 0.70 = tổng cả 3 tập
    train_counts = []
    for flower in FLOWERS:
        train_counts.append(len(os.listdir(f'{TRAIN_DIR}/{flower}')))
    total = round(sum(train_counts) / len(train_counts) / 0.70)
    half = total // 2                              # nửa hoa ngoài 5 loài, nửa không phải hoa

    print('Đang tải 3 bộ ảnh (khoảng 800 MB, vài phút)…')
    oxford     = load_by_class('dpdl-benchmark/oxford_flowers102', OXFORD_SKIP)   # Oxford 102 Flowers
    caltech    = load_by_class('flwrlabs/caltech101', CALTECH_SKIP)              # Caltech-101
    imagenette = load_by_class('johnowhitaker/imagenette2-320', [])              # Imagenette 320px

    # Gộp Caltech và Imagenette thành nhóm "không phải hoa"
    not_flower = {}
    not_flower.update(caltech)
    not_flower.update(imagenette)
    print(f'Hoa ngoài 5 loài: {len(oxford)} loài | Không phải hoa: {len(not_flower)} nhóm | Lớp Khác: {total} ảnh')

    rng = random.Random(42)   # bản chia cố định, không đổi theo SEED (giống notebook 01)

    # Mỗi phần tử là 1 cặp (loại, ảnh). Loại sẽ nằm trong tên file ("hoakhac" / "khonghoa") để Cell 9 đo riêng từng nửa.
    # Giữ đúng thứ tự: lấy Oxford trước rồi mới lấy nhóm không phải hoa, để rng chọn ra đúng các ảnh như cũ
    images = []
    for data in sample_evenly(oxford, half, rng):
        images.append(('hoakhac', data))
    for data in sample_evenly(not_flower, total - half, rng):
        images.append(('khonghoa', data))
    rng.shuffle(images)

    # Chia 15% test, 15% val, còn lại (70%) là train
    n = int(len(images) * 0.15)
    parts = {
        'test':  images[:n],
        'val':   images[n:2 * n],
        'train': images[2 * n:],
    }
    for split in parts:
        folder = f'{OTHER_DIR}/{split}/other'
        os.makedirs(folder, exist_ok=True)
        items = parts[split]
        for i in range(len(items)):
            kind, data = items[i]
            number = str(i).zfill(4)                       # 7 -> '0007'
            path = f'{folder}/{kind}_{split}_{number}.jpg'
            # Lưu lại thành JPEG màu RGB: Caltech có ảnh xám, Imagenette có vài ảnh CMYK mà TensorFlow không đọc được
            img = Image.open(io.BytesIO(data)).convert('RGB')
            img.save(path, quality=95)

    # Nén lại lên Drive. make_archive tự thêm đuôi .zip nên phải bỏ đuôi .zip khỏi tên trước
    shutil.make_archive(OTHER_ZIP.replace('.zip', ''), 'zip', OTHER_DIR)

# train, val của v2 thêm thư mục other/; test = TF test (5 loài, đúng tập đã chấm v1) + test của lớp Khác
shutil.copytree(f'{OTHER_DIR}/train/other', f'{TRAIN_DIR}/other', dirs_exist_ok=True)
shutil.copytree(f'{OTHER_DIR}/val/other',   f'{VAL_DIR}/other',   dirs_exist_ok=True)
shutil.copytree(f'{OTHER_DIR}/test/other',  f'{TEST_DIR}/other',  dirs_exist_ok=True)

counts = {
    'train': count(TRAIN_DIR),
    'val':   count(VAL_DIR),
    'test':  count(TEST_DIR),
}
print(json.dumps(counts, indent=2))

Cell 5 – Tạo tf.data, lưu class_names.json

In [ ]:
def load_dir(folder, class_names, shuffle):
    # class_names: đặt thứ tự lớp bằng tay. Để Keras tự xếp theo ABC thì "other" sẽ nằm giữa dandelion và roses
    return keras.utils.image_dataset_from_directory(
        folder, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
        class_names=class_names, shuffle=shuffle, seed=SEED)

train_ds    = load_dir(TRAIN_DIR, CLASS_NAMES, shuffle=True)
val_ds      = load_dir(VAL_DIR,   CLASS_NAMES, shuffle=False)
test_ds     = load_dir(TEST_DIR,  CLASS_NAMES, shuffle=False)
web_test_ds = load_dir(WEB_TEST_DIR, FLOWERS,  shuffle=False)   # chỉ 5 loài: nhãn 0–4 trùng CLASS_NAMES vì "other" ở cuối
test_files  = test_ds.file_paths                                # Cell 9 dùng để tách 2 nửa của lớp Khác
class_names = train_ds.class_names
print(class_names)

with open(f'{MODEL_DIR}/class_names.json', 'w') as f:
    json.dump(class_names, f)

AUTOTUNE = tf.data.AUTOTUNE
train_ds    = train_ds.cache().shuffle(1000, seed=SEED).prefetch(AUTOTUNE)
val_ds      = val_ds.cache().prefetch(AUTOTUNE)
test_ds     = test_ds.cache().prefetch(AUTOTUNE)
web_test_ds = web_test_ds.cache().prefetch(AUTOTUNE)

Cell 6 – Xây mô hình CNN (giống hệt v1, v2; chỉ khác lớp cuối là Softmax 6)

In [ ]:
L2 = regularizers.l2(1e-4)   # phạt trọng số lớn (mục 3 – Regularization)

model = keras.Sequential([
    layers.Input(shape=(180, 180, 3)),     # ảnh 180×180 (IMG_SIZE), 3 kênh màu RGB
    layers.Rescaling(1./255),              # giá trị điểm ảnh 0–255 -> 0–1

    # Tăng cường dữ liệu: chỉ chạy lúc huấn luyện, mỗi epoch ảnh bị biến đổi khác nhau
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomHue(0.1, value_range=(0, 1)),                  # xoay sắc độ màu một chút (đỏ ↔ hồng ↔ cam)
    layers.RandomSaturation((0.3, 0.7), value_range=(0, 1)),    # màu đậm/nhạt ngẫu nhiên
    layers.RandomGrayscale(0.2),                                # 20% ảnh thành trắng đen → phải nhìn hình dạng

    # Khối 1
    layers.Conv2D(16, 3, padding='same', activation='relu', kernel_regularizer=L2),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    # Khối 2
    layers.Conv2D(32, 3, padding='same', activation='relu', kernel_regularizer=L2),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    # Khối 3
    layers.Conv2D(64, 3, padding='same', activation='relu', kernel_regularizer=L2),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    # Khối 4
    layers.Conv2D(128, 3, padding='same', activation='relu', kernel_regularizer=L2),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.GlobalAveragePooling2D(),       # thay Flatten + Dense(128): mỗi kênh lấy trung bình còn 1 số
    layers.Dropout(0.2),
    layers.Dense(len(class_names), activation='softmax'),   # 6 lớp: 5 loài hoa + Khác
], name='flower_cnn')

model.summary()

Cell 7 – Huấn luyện

In [ ]:
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

callbacks = [
    # Dừng sớm nếu val_loss không giảm suốt 20 epoch
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True),
    # val_loss đứng yên 5 epoch -> giảm learning rate còn một nửa
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-5),
    # Lưu lại mô hình mỗi khi val_loss thấp hơn trước
    keras.callbacks.ModelCheckpoint(CKPT_PATH, monitor='val_loss', save_best_only=True),
]
history = model.fit(train_ds, validation_data=val_ds, epochs=100, callbacks=callbacks)

model = keras.models.load_model(CKPT_PATH)   # dùng bản có val_loss thấp nhất

# Lưu số liệu từng epoch ngay, phòng khi phiên Colab bị ngắt trước Cell 10
history_data = {}
for key in history.history:                  # key: 'accuracy', 'loss', 'val_accuracy', ...
    values = []
    for v in history.history[key]:
        values.append(float(v))
    history_data[key] = values

with open(f'{ROOT}/checkpoints/history_khac_v2.json', 'w') as f:
    json.dump(history_data, f)

Cell 8 – Biểu đồ accuracy / loss theo epoch

In [ ]:
h = history.history
epochs = range(1, len(h['loss']) + 1)        # 1, 2, 3, ..., số epoch đã chạy

plt.figure(figsize=(12, 4))

# Biểu đồ trái: accuracy
plt.subplot(1, 2, 1)
plt.plot(epochs, h['accuracy'], label='Train')
plt.plot(epochs, h['val_accuracy'], label='Validation')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.legend()

# Biểu đồ phải: loss
plt.subplot(1, 2, 2)
plt.plot(epochs, h['loss'], label='Train')
plt.plot(epochs, h['val_loss'], label='Validation')
plt.title('Loss')
plt.xlabel('Epoch')
plt.legend()

plt.tight_layout()
plt.savefig(f'{MODEL_DIR}/training_curves.png', dpi=150)
plt.show()

# Epoch tốt nhất = epoch có val_loss nhỏ nhất (khớp với epoch mà checkpoint đã lưu). Epoch đếm từ 1 nên + 1
best_epoch = h['val_loss'].index(min(h['val_loss'])) + 1
gap = h['accuracy'][best_epoch - 1] - h['val_accuracy'][best_epoch - 1]
print(f'Epoch tốt nhất: {best_epoch}/{len(h["loss"])} | train – val accuracy = {gap:.3f}')
if gap > 0.15:
    print('⚠ Overfitting: tăng Dropout lên 0.3–0.4 hoặc tăng mức tăng cường dữ liệu')

Cell 9 – Đánh giá: tập test 6 lớp, riêng 5 loài hoa, riêng lớp Khác, ảnh thực tế

In [ ]:
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f'Test accuracy (6 lớp: TF test + Khác) = {test_acc:.4f}')

probs  = model.predict(test_ds, verbose=0)   # mỗi dòng: 6 xác suất của 1 ảnh
y_pred = probs.argmax(axis=1)                # lớp đoán = lớp có xác suất cao nhất

# Nhãn thật: lấy từ test_ds, từng batch một (cùng thứ tự với y_pred vì test_ds không xáo trộn)
y_true = []
for images, labels in test_ds:
    y_true.extend(labels.numpy())
y_true = np.array(y_true)

OTHER = class_names.index('other')   # = 5

print(classification_report(y_true, y_pred, target_names=class_names, digits=3, zero_division=0))
report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True, zero_division=0)

# 5 loài hoa trên đúng tập TF test: so được với v1 (80,4%), v2 (83,2%), v1-khac (83,0%)
num_flower = 0          # số ảnh hoa (không phải lớp Khác)
flower_correct = 0      # số ảnh hoa đoán đúng loài
flower_as_other = 0     # số ảnh hoa bị đoán nhầm thành Khác
for i in range(len(y_true)):
    if y_true[i] != OTHER:
        num_flower += 1
        if y_pred[i] == y_true[i]:
            flower_correct += 1
        if y_pred[i] == OTHER:
            flower_as_other += 1
flower_acc = flower_correct / num_flower
print(f'5 loài hoa (TF test): accuracy {flower_acc:.1%} | bị đoán nhầm thành Khác {flower_as_other / num_flower:.1%}')

# Lớp Khác, tách 2 nửa theo tên file (Cell 4): "hoakhac_..." và "khonghoa_..."
for prefix, label in [('hoakhac_', 'Hoa ngoài 5 loài'), ('khonghoa_', 'Không phải hoa')]:
    num_images = 0
    num_found = 0       # số ảnh mô hình nhận ra là Khác
    for i in range(len(test_files)):
        file_name = os.path.basename(test_files[i])
        if file_name.startswith(prefix):
            num_images += 1
            if y_pred[i] == OTHER:
                num_found += 1
    print(f'{label.ljust(17)}: nhận ra là Khác {num_found / num_images:.1%} ({num_images} ảnh)')

# Quy tắc trên web: đứng đầu là Khác -> "không phải hoa"; độ tin cậy dưới ngưỡng -> "chưa chắc chắn".
# Đếm xem bao nhiêu ảnh lớp Khác bị web chặn lại theo 1 trong 2 cách đó
num_other = 0
num_blocked = 0
for i in range(len(y_true)):
    if y_true[i] == OTHER:
        num_other += 1
        if y_pred[i] == OTHER or probs[i].max() < THRESHOLD:
            num_blocked += 1
print(f'Ảnh Khác bị web chặn (Khác hoặc chưa chắc chắn): {num_blocked / num_other:.1%}')

# Ảnh thực tế (web test, 5 loài): so được với v1 (63,1%), v2 (86,6%), v1-khac (54,0%)
web_pred = model.predict(web_test_ds, verbose=0).argmax(axis=1)
web_true = []
for images, labels in web_test_ds:
    web_true.extend(labels.numpy())

web_correct = 0
web_as_other = 0
for i in range(len(web_true)):
    if web_pred[i] == web_true[i]:
        web_correct += 1
    if web_pred[i] == OTHER:
        web_as_other += 1
web_acc = web_correct / len(web_true)
print(f'Ảnh thực tế (web test): accuracy {web_acc:.1%} | bị đoán nhầm thành Khác {web_as_other / len(web_true):.1%}')

# Ma trận nhầm lẫn: hàng = lớp thật, cột = lớp đoán
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(7, 7))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=ax, cmap='Blues', colorbar=False)
plt.title(f'Confusion matrix – test accuracy {test_acc:.1%}')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(f'{MODEL_DIR}/confusion_matrix.png', dpi=150)
plt.show()

Cell 10 – Xuất file cho web và kiểm tra nạp lại

In [ ]:
model.save(f'{MODEL_DIR}/flower_cnn.keras')

# Precision / recall / f1 của từng lớp, làm tròn 4 chữ số
per_class = {}
for c in class_names:
    per_class[c] = {
        'precision': round(report[c]['precision'], 4),
        'recall':    round(report[c]['recall'], 4),
        'f1':        round(report[c]['f1-score'], 4),
    }

# Số liệu từng epoch, làm tròn 4 chữ số (web vẽ lại biểu đồ từ đây)
history_rounded = {}
for key in ['accuracy', 'val_accuracy', 'loss', 'val_loss']:
    values = []
    for v in h[key]:
        values.append(round(float(v), 4))
    history_rounded[key] = values

model_info = {
    'version': VERSION,
    'uncertain_threshold': THRESHOLD,
    'charts': {
        'training_curves': '/static/model/training_curves.png',
        'confusion_matrix': '/static/model/confusion_matrix.png',
    },
    'img_size': list(IMG_SIZE),
    'class_names': class_names,
    'architecture': [
        'Input 180×180×3',
        'Rescaling 1/255',
        'Augment: Flip · Rotation · Zoom',
        'Augment màu: Hue · Saturation · Grayscale',
        'Conv 16 + L2 + BN + Pool',
        'Conv 32 + L2 + BN + Pool',
        'Conv 64 + L2 + BN + Pool',
        'Conv 128 + L2 + BN + Pool',
        'GlobalAveragePooling',
        'Dropout 0.2',
        'Softmax 6',
    ],
    'num_images': counts,
    'test_accuracy': round(float(test_acc), 4),
    'flower_test_accuracy': round(float(flower_acc), 4),     # riêng 5 loài trên TF test, để so với v1, v2
    'web_test_accuracy': round(float(web_acc), 4),           # ảnh thực tế (web test)
    'per_class': per_class,
    'confusion_matrix': cm.tolist(),                          # hàng = thực tế, cột = dự đoán
    'history': history_rounded,
    'epochs_trained': len(h['loss']),
    'best_epoch': best_epoch,
    'trained_at': datetime.date.today().isoformat(),
    'tensorflow_version': tf.__version__,
}
with open(f'{MODEL_DIR}/model_info.json', 'w', encoding='utf-8') as f:
    json.dump(model_info, f, ensure_ascii=False, indent=2)

# Kiểm tra file .keras nạp lại được và cho kết quả giống hệt (chênh lệch lớn nhất phải là 0)
loaded = keras.models.load_model(f'{MODEL_DIR}/flower_cnn.keras')
new_probs = loaded.predict(test_ds, verbose=0)
print('Chênh lệch sau khi nạp lại:', np.abs(new_probs - probs).max())

# Liệt kê các file đã lưu và dung lượng
for file_name in sorted(os.listdir(MODEL_DIR)):
    size_mb = os.path.getsize(f'{MODEL_DIR}/{file_name}') / 1e6
    print(f'{file_name.ljust(22)} {size_mb:.1f} MB')

Cell 11 – Thử phân loại 1 ảnh ngoài bộ dữ liệu (thử cả ảnh không phải hoa)

In [ ]:
from google.colab import files

model_test = keras.models.load_model(f'{MODEL_DIR}/flower_cnn.keras')
with open(f'{MODEL_DIR}/class_names.json') as f:
    names = json.load(f)

uploaded = files.upload()               # chọn 1 ảnh JPG/PNG. Kết quả: {tên file: nội dung dạng bytes}
fname = list(uploaded.keys())[0]        # chỉ lấy file đầu tiên
data = uploaded[fname]

img = tf.io.decode_image(data, channels=3, expand_animations=False)   # bytes -> ảnh (cao, rộng, 3)
img_small = tf.image.resize(img, IMG_SIZE)                            # đổi cỡ về 180×180
batch = tf.expand_dims(img_small, 0)                                  # mô hình nhận cả lô ảnh: (1, 180, 180, 3)
p = model_test.predict(batch, verbose=0)[0]                           # 6 xác suất của ảnh này

# Lấy 3 lớp có xác suất cao nhất
order = p.argsort()       # chỉ số lớp, xếp theo xác suất TĂNG dần
order = order[::-1]       # đảo ngược -> GIẢM dần
top3 = order[:3]
best = top3[0]

print(f'Kết quả: {names[best]} ({p[best]:.1%})')
top3_text = []
for i in top3:
    top3_text.append(f'{names[i]} {p[i]:.1%}')
print('Top 3  :', ', '.join(top3_text))

if names[best] == 'other':
    print('→ Web sẽ báo: không phải 1 trong 5 loài hoa Anthea nhận diện')
elif p[best] < THRESHOLD:
    print('→ Web sẽ báo: chưa chắc chắn')

plt.imshow(img.numpy())
plt.title(f'{names[best]} ({p[best]:.0%})')
plt.axis('off')
plt.show()